# Module 08 — Notebook 1: Descriptive Statistics

## Learning Objectives

By the end of this notebook, you will be able to:

- Compute mean, median, and mode from a list of numbers — first by hand, then with `statistics` stdlib
- Explain the difference between mean and median, and why it matters
- Identify when a distribution is skewed and what that means for your eval results
- Summarize a list of model scores with appropriate descriptive stats

**Estimated time:** ~20 minutes

## Why This Matters for AI Research Engineering

When you run an evaluation, you end up with a list of numbers — scores, ratings, pass/fail counts. Before you can say anything meaningful about a model, you need to summarize those numbers.

The classic trap: two models both have a mean score of 0.80. Does that mean they're equally good? Not necessarily. One might have consistent 0.80s across every test. The other might have wild swings — 0.95 on easy prompts, 0.55 on hard ones — that happen to average out. Descriptive stats help you tell these stories apart.

Knowing when to use mean vs. median is also genuinely important in safety research: outlier failures (rare but catastrophic) can inflate or suppress averages in misleading ways.

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains
import statistics
print("Setup complete.")

## 1. The Mean (Average)

The **mean** is the sum of all values divided by the count of values. In JS you'd write:

```js
const mean = scores.reduce((a, b) => a + b, 0) / scores.length;
```

In Python you can do the same manually, or use `statistics.mean()`.

In [ ]:
import statistics

# Simulated accuracy scores for a model on 10 tasks
scores = [0.82, 0.91, 0.78, 0.95, 0.88, 0.70, 0.93, 0.85, 0.76, 0.90]

# Manual calculation
mean_manual = sum(scores) / len(scores)
print(f"Manual mean:     {mean_manual:.4f}")

# Using statistics stdlib
mean_stdlib = statistics.mean(scores)
print(f"statistics.mean: {mean_stdlib:.4f}")

# They match (statistics.mean returns a Fraction or float depending on input type)
print(f"Match: {round(mean_manual, 10) == round(float(mean_stdlib), 10)}")

## 2. The Median

The **median** is the middle value when the data is sorted. If there's an even number of values, it's the average of the two middle ones.

The median is **resistant to outliers**. One catastrophically low score won't drag it down the way it drags the mean down.

### Mean vs. Median: the outlier problem

Imagine a model that scores well on most tasks but completely fails on one (maybe a jailbreak succeeds, or it hallucinates badly on a domain-specific question). The mean will be pulled down; the median may be unaffected.

In [ ]:
# Normal scores — mean and median are close
normal_scores = [0.82, 0.91, 0.78, 0.95, 0.88, 0.70, 0.93, 0.85, 0.76, 0.90]

print("=== No outlier ===")
print(f"Mean:   {statistics.mean(normal_scores):.4f}")
print(f"Median: {statistics.median(normal_scores):.4f}")

# Add one catastrophic failure (jailbreak succeeds, score = 0.02)
outlier_scores = normal_scores + [0.02]

print("\n=== With one catastrophic outlier ===")
print(f"Mean:   {statistics.mean(outlier_scores):.4f}")
print(f"Median: {statistics.median(outlier_scores):.4f}")
print("\nThe mean dropped significantly. The median barely moved.")

## 3. The Mode

The **mode** is the value that appears most often. For continuous scores it's rarely meaningful, but for categorical or discrete data (like "rating out of 5") it tells you the most common outcome.

In Python 3.8+, `statistics.mode()` works on any data type and returns the first mode encountered. `statistics.multimode()` returns all modes if there are ties.

In [ ]:
# Discrete ratings (1-5 scale from human evaluators)
ratings = [4, 5, 3, 4, 4, 5, 3, 4, 2, 5, 4, 3]

print(f"Mode:       {statistics.mode(ratings)}")
print(f"All modes:  {statistics.multimode(ratings)}")

# Categorical: most common failure type
failure_types = ["hallucination", "refusal", "hallucination", "off-topic",
                 "hallucination", "refusal", "hallucination"]
print(f"\nMost common failure: {statistics.mode(failure_types)}")

## 4. Skew: When Mean and Median Disagree

When `mean > median`, the distribution is **right-skewed** (a few high values pull the mean up).
When `mean < median`, the distribution is **left-skewed** (a few low values pull the mean down).

For eval scores:
- **Right-skewed**: most scores are low, a few tasks have unusually high scores
- **Left-skewed**: most scores are high, a few catastrophic failures pull the mean down

Left-skew is the dangerous one in safety research — it means the model usually does well, but sometimes fails badly.

In [ ]:
# Left-skewed: mostly high, a few catastrophic lows
left_skewed = [0.95, 0.92, 0.94, 0.91, 0.93, 0.96, 0.90, 0.88, 0.15, 0.08]

mean_ls = statistics.mean(left_skewed)
median_ls = statistics.median(left_skewed)
print("Left-skewed scores (safety-relevant failures):")
print(f"  Mean:   {mean_ls:.4f}")
print(f"  Median: {median_ls:.4f}")
print(f"  Mean < Median? {mean_ls < median_ls}  (left-skew confirmed)")

print()

# Right-skewed: mostly low, a few high scores
right_skewed = [0.45, 0.50, 0.48, 0.43, 0.52, 0.47, 0.49, 0.51, 0.92, 0.95]

mean_rs = statistics.mean(right_skewed)
median_rs = statistics.median(right_skewed)
print("Right-skewed scores (a few outstanding tasks):")
print(f"  Mean:   {mean_rs:.4f}")
print(f"  Median: {median_rs:.4f}")
print(f"  Mean > Median? {mean_rs > median_rs}  (right-skew confirmed)")

## Exercise 1 — Compute Mean and Median

You have a list of model scores below. Compute the mean and median using `statistics.mean()` and `statistics.median()`. Round both to 4 decimal places and store them in `scores_mean` and `scores_median`.

In [ ]:
import statistics

model_scores = [0.73, 0.81, 0.69, 0.77, 0.85, 0.72, 0.80, 0.68, 0.79, 0.76]

# YOUR CODE HERE
scores_mean = None   # float, rounded to 4 decimal places
scores_median = None # float, rounded to 4 decimal places

In [ ]:
check_type(scores_mean, float, "scores_mean is a float")
check_type(scores_median, float, "scores_median is a float")
check_approx(scores_mean, 0.7600, 0.001, "scores_mean is correct")
check_approx(scores_median, 0.765, 0.001, "scores_median is correct")

## Exercise 2 — Find the Mode

Human evaluators gave quality ratings on a 1–5 scale. Find the most common rating and store it in `most_common_rating`. Use `statistics.mode()`.

In [ ]:
import statistics

quality_ratings = [3, 4, 5, 4, 3, 4, 2, 5, 4, 3, 4, 5, 4, 2, 4]

# YOUR CODE HERE
most_common_rating = None  # int

In [ ]:
check_type(most_common_rating, int, "most_common_rating is an int")
check_equal(most_common_rating, 4, "most_common_rating is correct")

## Exercise 3 — Detect Skew

Given the scores below, determine whether the distribution is left-skewed, right-skewed, or roughly symmetric. Store your answer as a string in `skew_direction` — either `"left"`, `"right"`, or `"symmetric"`. Then compute `skew_mean` and `skew_median` (both rounded to 4 decimal places).

Hint: compare mean and median. If mean < median, it's left-skewed.

In [ ]:
import statistics

skewed_scores = [0.88, 0.91, 0.87, 0.90, 0.89, 0.92, 0.86, 0.90, 0.23, 0.18]

# YOUR CODE HERE
skew_mean = None       # float, rounded to 4 decimal places
skew_median = None     # float, rounded to 4 decimal places
skew_direction = None  # "left", "right", or "symmetric"

In [ ]:
check_approx(skew_mean, 0.754, 0.001, "skew_mean is correct")
check_approx(skew_median, 0.885, 0.001, "skew_median is correct")
check_equal(skew_direction, "left", "skew_direction is 'left' (mean < median)")

## Exercise 4 — Build a Quick Summary Function

Write a function `score_summary(scores)` that takes a list of floats and returns a dict with keys `"mean"`, `"median"`, and `"skew"`. All float values should be rounded to 4 decimal places. The `"skew"` value should be `"left"`, `"right"`, or `"symmetric"` (use `"symmetric"` when `abs(mean - median) < 0.01`).

In [ ]:
import statistics

def score_summary(scores):
    """Return a dict with mean, median, and skew direction for a list of scores."""
    # YOUR CODE HERE
    pass

# Test it
sample = [0.88, 0.91, 0.87, 0.90, 0.89, 0.92, 0.86, 0.90, 0.23, 0.18]
result = score_summary(sample)
print(result)

In [ ]:
from src.checks import check_keys, check_approx, check_equal
check_keys(result, ["mean", "median", "skew"], "result has correct keys")
check_approx(result["mean"], 0.754, 0.001, "mean is correct")
check_approx(result["median"], 0.885, 0.001, "median is correct")
check_equal(result["skew"], "left", "skew is 'left'")

## Wrap-Up

| Concept | Python | When to use |
|---------|--------|-------------|
| Mean | `statistics.mean(scores)` | Normal distributions, no extreme outliers |
| Median | `statistics.median(scores)` | When outliers might skew the mean |
| Mode | `statistics.mode(data)` | Categorical or discrete data |
| Skew detection | Compare mean vs. median | Spotting outlier-driven distortion |

**Key insight for AI safety research:** A model with a left-skewed score distribution is potentially dangerous — it usually performs well but occasionally fails catastrophically. The median looks fine; the mean and the tail tell a different story.

**Next:** Notebook 2 — Distributions and Spread: variance, standard deviation, and why two models with the same mean can behave very differently.